
## Preprocessing Strategy
This notebook prepares the data for ensemble classifiers by:
1. Loading and inspecting the dataset
2. Engineering fraud-relevant features
3. Encoding categorical variables
4. Handling class imbalance
5. Preparing train/test splits
6. Saving preprocessed data for model training

In [18]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle
import os

print("Libraries imported successfully")

Libraries imported successfully


## Step 1: Load and Inspect Data

In [ ]:
# Load the historical fraud data from cybersecurity team
df = pd.read_csv("../data/dirty_data.csv")


df.head()

=== Caishen Bank Fraud Detection Dataset ===

Dataset shape: (6362620, 11)
Total transactions: 6,362,620

Column names:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

First few records:


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [ ]:
# Examine data types and missing values
print("=== Data Quality Check ===")
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())


=== Data Quality Check ===

Data types:
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Total missing values: 0


In [21]:
# Understand the fraud distribution (target variable)
print("=== Fraud Distribution Analysis ===")
print(f"\nTarget variable: isFraud")
print(df['isFraud'].value_counts())
fraud_rate = (df['isFraud'].sum() / len(df)) * 100
print(f"\nFraud rate: {fraud_rate:.4f}%")
print(f"Total fraudulent transactions: {df['isFraud'].sum():,}")
print(f"Total legitimate transactions: {(df['isFraud'] == 0).sum():,}")

# Analyze fraud by transaction type
print("\nFraud by transaction type:")
fraud_by_type = df.groupby('type')['isFraud'].agg(['sum', 'count', 'mean'])
fraud_by_type.columns = ['Fraud Count', 'Total', 'Fraud Rate']
fraud_by_type['Fraud Rate'] = fraud_by_type['Fraud Rate'] * 100
print(fraud_by_type)

=== Fraud Distribution Analysis ===

Target variable: isFraud
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud rate: 0.1291%
Total fraudulent transactions: 8,213
Total legitimate transactions: 6,354,407

Fraud by transaction type:
          Fraud Count    Total  Fraud Rate
type                                      
CASH_IN             0  1399284    0.000000
CASH_OUT         4116  2237500    0.183955
DEBIT               0    41432    0.000000
PAYMENT             0  2151495    0.000000
TRANSFER         4097   532909    0.768799


## Step 2: Feature Engineering

Creating features that will help ensemble classifiers detect fraud patterns:

In [22]:
print("=== Feature Engineering for Fraud Detection ===")

# Create a copy for preprocessing
df_processed = df.copy()

# 1. Balance change features (important for detecting unusual money movements)
df_processed['balance_change_orig'] = df_processed['newbalanceOrig'] - df_processed['oldbalanceOrg']
df_processed['balance_change_dest'] = df_processed['newbalanceDest'] - df_processed['oldbalanceDest']

# 2. Amount to balance ratio (detects transactions unusual for account size)
df_processed['amount_to_balance_ratio'] = df_processed['amount'] / (df_processed['oldbalanceOrg'] + 1)

# 3. Zero balance flags (accounts with $0 balance may indicate fraud)
df_processed['is_zero_balance_orig'] = (df_processed['oldbalanceOrg'] == 0).astype(int)
df_processed['is_zero_balance_dest'] = (df_processed['oldbalanceDest'] == 0).astype(int)

# 4. Balance error detection (catches inconsistent transaction records)
# For origin: oldbalance - amount should equal newbalance
df_processed['balance_error_orig'] = abs(
    (df_processed['oldbalanceOrg'] - df_processed['amount']) - df_processed['newbalanceOrig']
)

# For destination: oldbalance + amount should equal newbalance  
df_processed['balance_error_dest'] = abs(
    (df_processed['oldbalanceDest'] + df_processed['amount']) - df_processed['newbalanceDest']
)

# Flag significant balance errors (> $0.01 difference)
df_processed['has_balance_error'] = (
    (df_processed['balance_error_orig'] > 0.01) | 
    (df_processed['balance_error_dest'] > 0.01)
).astype(int)

# 5. Extract account type from customer IDs
# C = Customer account, M = Merchant account
df_processed['account_type_orig'] = df_processed['nameOrig'].str[0]
df_processed['account_type_dest'] = df_processed['nameDest'].str[0]

# 6. Customer-to-customer transaction flag (higher fraud risk)
df_processed['is_c2c_transaction'] = (
    (df_processed['account_type_orig'] == 'C') & 
    (df_processed['account_type_dest'] == 'C')
).astype(int)

print("\n✓ Engineered features created:")
print("  - balance_change_orig")
print("  - balance_change_dest")
print("  - amount_to_balance_ratio")
print("  - is_zero_balance_orig")
print("  - is_zero_balance_dest")
print("  - balance_error_orig")
print("  - balance_error_dest")
print("  - has_balance_error")
print("  - account_type_orig")
print("  - account_type_dest")
print("  - is_c2c_transaction")

print(f"\nDataset shape after feature engineering: {df_processed.shape}")

=== Feature Engineering for Fraud Detection ===

✓ Engineered features created:
  - balance_change_orig
  - balance_change_dest
  - amount_to_balance_ratio
  - is_zero_balance_orig
  - is_zero_balance_dest
  - balance_error_orig
  - balance_error_dest
  - has_balance_error
  - account_type_orig
  - account_type_dest
  - is_c2c_transaction

Dataset shape after feature engineering: (6362620, 22)


## Step 3: Encode Categorical Variables

For ensemble models (Random Forest, Gradient Boosting), we'll use Label Encoding which is efficient and works well with tree-based models.

In [23]:
print("=== Encoding Categorical Variables ===")

# Encode transaction type
le_type = LabelEncoder()
df_processed['type_encoded'] = le_type.fit_transform(df_processed['type'])

# Encode account types
le_account_orig = LabelEncoder()
le_account_dest = LabelEncoder()
df_processed['account_type_orig_encoded'] = le_account_orig.fit_transform(df_processed['account_type_orig'])
df_processed['account_type_dest_encoded'] = le_account_dest.fit_transform(df_processed['account_type_dest'])

print("\nEncoding mappings:")
print(f"\nTransaction types: {dict(zip(le_type.classes_, le_type.transform(le_type.classes_)))}")
print(f"Account type (origin): {dict(zip(le_account_orig.classes_, le_account_orig.transform(le_account_orig.classes_)))}")
print(f"Account type (dest): {dict(zip(le_account_dest.classes_, le_account_dest.transform(le_account_dest.classes_)))}")

print("\n✓ Categorical encoding complete")

=== Encoding Categorical Variables ===

Encoding mappings:

Transaction types: {'CASH_IN': np.int64(0), 'CASH_OUT': np.int64(1), 'DEBIT': np.int64(2), 'PAYMENT': np.int64(3), 'TRANSFER': np.int64(4)}
Account type (origin): {'C': np.int64(0)}
Account type (dest): {'C': np.int64(0), 'M': np.int64(1)}

✓ Categorical encoding complete


## Step 4: Remove Irrelevant Features

In [24]:
print("=== Removing Irrelevant Features ===")

# Features to remove:
# - isFlaggedFraud: Only caught 16 out of 8,213 frauds (ineffective)
# - nameOrig/nameDest: Customer IDs (not predictive, privacy concern)
# - account_type_orig/dest: Already encoded
# - step: Time step (keeping for now, can remove if not useful in model)
# - type: Already encoded

columns_to_drop = [
    'isFlaggedFraud',
    'nameOrig',
    'nameDest',
    'account_type_orig',
    'account_type_dest',
    'type'
]

print(f"\nDropping columns: {columns_to_drop}")
df_processed = df_processed.drop(columns=columns_to_drop)

print(f"\n✓ Irrelevant features removed")
print(f"Dataset shape: {df_processed.shape}")
print(f"\nRemaining features:")
print([col for col in df_processed.columns if col != 'isFraud'])

=== Removing Irrelevant Features ===

Dropping columns: ['isFlaggedFraud', 'nameOrig', 'nameDest', 'account_type_orig', 'account_type_dest', 'type']

✓ Irrelevant features removed
Dataset shape: (6362620, 19)

Remaining features:
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'balance_change_orig', 'balance_change_dest', 'amount_to_balance_ratio', 'is_zero_balance_orig', 'is_zero_balance_dest', 'balance_error_orig', 'balance_error_dest', 'has_balance_error', 'is_c2c_transaction', 'type_encoded', 'account_type_orig_encoded', 'account_type_dest_encoded']


## Step 5: Prepare Features and Target

In [25]:
print("=== Separating Features and Target ===")

# Target variable (what we're predicting)
y = df_processed['isFraud']

# Features (predictors)
X = df_processed.drop('isFraud', axis=1)

print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"\nNumber of features: {X.shape[1]}")
print(f"\nClass distribution:")
print(y.value_counts())
print(f"\nClass imbalance ratio: {y.value_counts()[0] / y.value_counts()[1]:.1f}:1 (legitimate:fraud)")

=== Separating Features and Target ===

Features (X) shape: (6362620, 18)
Target (y) shape: (6362620,)

Number of features: 18

Class distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Class imbalance ratio: 773.7:1 (legitimate:fraud)


## Step 6: Train-Test Split (Stratified)

Using stratified split to maintain the fraud rate in both train and test sets.

In [ ]:
print("=== Creating Train-Test Split ===")

# 80/20 split with stratification to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Ensures same fraud percentage in train and test
)

print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

print(f"\nTraining set fraud distribution:")
print(y_train.value_counts())
print(f"Fraud rate: {(y_train.sum() / len(y_train) * 100):.4f}%")

print(f"\nTest set fraud distribution:")
print(y_test.value_counts())
print(f"Fraud rate: {(y_test.sum() / len(y_test) * 100):.4f}%")



=== Creating Train-Test Split ===

Training set size: (5090096, 18)
Test set size: (1272524, 18)

Training set fraud distribution:
isFraud
0    5083526
1       6570
Name: count, dtype: int64
Fraud rate: 0.1291%

Test set fraud distribution:
isFraud
0    1270881
1       1643
Name: count, dtype: int64
Fraud rate: 0.1291%

✓ Data split complete - ready for ensemble model training


## Step 7: Save Preprocessed Data

**Note on Scaling:** For tree-based ensemble models (Random Forest, Gradient Boosting), feature scaling is NOT required as these models are invariant to monotonic transformations. We'll save the unscaled data for ensemble classifiers.

**Note on Class Imbalance:** We'll handle class imbalance during model training using:
- `class_weight='balanced'` parameter in sklearn models
- Or SMOTE if needed for specific models

In [ ]:
print("=== Saving Preprocessed Data ===")

# Create directory for preprocessed data
os.makedirs('../data/preprocessed', exist_ok=True)

# Save training data
X_train.to_csv('../data/preprocessed/X_train.csv', index=False)
y_train.to_csv('../data/preprocessed/y_train.csv', index=False, header=True)

# Save test data
X_test.to_csv('../data/preprocessed/X_test.csv', index=False)
y_test.to_csv('../data/preprocessed/y_test.csv', index=False, header=True)

# Save label encoders for future use (deployment)
with open('../data/preprocessed/label_encoders.pkl', 'wb') as f:
    pickle.dump({
        'type': le_type,
        'account_orig': le_account_orig,
        'account_dest': le_account_dest
    }, f)

# Save feature names
with open('../data/preprocessed/feature_names.pkl', 'wb') as f:
    pickle.dump(list(X_train.columns), f)



=== Saving Preprocessed Data ===

✓ Preprocessed data saved to '../data/preprocessed/'

Saved files:
  - X_train.csv (training features)
  - y_train.csv (training labels)
  - X_test.csv (test features)
  - y_test.csv (test labels)
  - label_encoders.pkl (encoding mappings)
  - feature_names.pkl (feature names)
